# Nitor Energy EDA Notebook

This notebook performs a thorough exploratory data analysis on:
- `data/train.csv`
- `data/test_for_participants.csv`

Focus areas:
1. Data integrity and missingness
2. Target behavior (overall, by market, over time)
3. Feature distributions and outlier diagnostics
4. Correlations and feature-target relationships
5. Temporal seasonality patterns
6. Train-vs-test drift checks
7. Engineered-feature diagnostics aligned with training scripts

In [ ]:
# If needed, install optional plotting helpers in this environment.
# Uncomment if a package import fails.
# %pip install seaborn missingno -q

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 220)

In [ ]:
BASE_DIR = Path('.')
DATA_DIR = BASE_DIR / 'data'
TRAIN_PATH = DATA_DIR / 'train.csv'
TEST_PATH = DATA_DIR / 'test_for_participants.csv'

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

for df in [train, test]:
    df['delivery_start'] = pd.to_datetime(df['delivery_start'], errors='coerce')
    df['delivery_end'] = pd.to_datetime(df['delivery_end'], errors='coerce')

train.shape, test.shape

In [ ]:
print('Train rows/cols:', train.shape)
print('Test rows/cols :', test.shape)
print()
print('Train dtypes:')
print(train.dtypes)

In [ ]:
print('Train sample:')
display(train.head(3))
print('Test sample:')
display(test.head(3))

## 1) Basic Structure and Time Coverage

In [ ]:
def coverage_summary(df, name):
    print(f'[{name}]')
    print('Unique markets:', df['market'].nunique())
    print('Start:', df['delivery_start'].min())
    print('End  :', df['delivery_start'].max())
    print('Unique timestamps:', df['delivery_start'].nunique())
    print('Rows per timestamp (mean):', round(df.groupby('delivery_start').size().mean(), 2))
    print('Rows per market:')
    display(df['market'].value_counts().sort_index().to_frame('rows'))

coverage_summary(train, 'TRAIN')
coverage_summary(test, 'TEST')

In [ ]:
rows_ts_train = train.groupby('delivery_start').size().rename('rows').reset_index()
fig = px.line(rows_ts_train, x='delivery_start', y='rows', title='Rows per timestamp in train')
fig.show()

## 2) Missing Data Analysis

In [ ]:
def missing_table(df):
    miss = df.isna().mean().sort_values(ascending=False)
    miss_count = df.isna().sum().reindex(miss.index)
    out = pd.DataFrame({'missing_ratio': miss, 'missing_count': miss_count})
    return out

miss_train = missing_table(train)
miss_test = missing_table(test)

print('Top missing columns (train):')
display(miss_train.head(15))
print('Top missing columns (test):')
display(miss_test.head(15))

In [ ]:
plot_df = miss_train.reset_index().rename(columns={'index': 'feature'})
plot_df = plot_df[plot_df['missing_ratio'] > 0]

if not plot_df.empty:
    fig = px.bar(
        plot_df,
        x='feature',
        y='missing_ratio',
        title='Missing ratio by feature (train)',
    )
    fig.update_layout(xaxis_tickangle=-45)
    fig.show()
else:
    print('No missing values in train.')

## 3) Target Analysis

In [ ]:
train['target'].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])

In [ ]:
fig = px.histogram(train, x='target', nbins=120, title='Target distribution (overall)', marginal='box')
fig.show()

In [ ]:
market_target = train.groupby('market')['target'].agg(['mean', 'std', 'min', 'median', 'max', 'count']).sort_values('mean')
display(market_target)

fig = px.box(train, x='market', y='target', title='Target by market (boxplot)', points=False)
fig.show()

In [ ]:
q_low, q_high = train['target'].quantile([0.01, 0.99])
trimmed = train[(train['target'] >= q_low) & (train['target'] <= q_high)].copy()
fig = px.violin(trimmed, x='market', y='target', box=True, points=False, title='Target by market (1%-99% trimmed)')
fig.show()

In [ ]:
train_ts = train.groupby('delivery_start')['target'].agg(['mean', 'median', 'std']).reset_index()
fig = go.Figure()
fig.add_trace(go.Scatter(x=train_ts['delivery_start'], y=train_ts['mean'], mode='lines', name='mean target'))
fig.add_trace(go.Scatter(x=train_ts['delivery_start'], y=train_ts['median'], mode='lines', name='median target'))
fig.update_layout(title='Target over time (global aggregates)', xaxis_title='delivery_start', yaxis_title='target')
fig.show()

In [ ]:
for m in sorted(train['market'].unique()):
    sub = train[train['market'] == m].sort_values('delivery_start')
    roll = sub.set_index('delivery_start')['target'].rolling(24, min_periods=12).mean().rename('target_24h_mean').reset_index()
    fig = px.line(roll, x='delivery_start', y='target_24h_mean', title=f'24h rolling mean target - {m}')
    fig.show()

## 4) Calendar and Seasonality Patterns

In [ ]:
for df in [train, test]:
    df['hour'] = df['delivery_start'].dt.hour
    df['dow'] = df['delivery_start'].dt.dayofweek
    df['month'] = df['delivery_start'].dt.month
    df['is_weekend'] = df['dow'].isin([5, 6]).astype(int)

hour_profile = train.groupby(['market', 'hour'])['target'].mean().reset_index()
fig = px.line(hour_profile, x='hour', y='target', color='market', title='Mean target by hour of day')
fig.show()

week_profile = train.groupby(['market', 'dow'])['target'].mean().reset_index()
fig = px.line(week_profile, x='dow', y='target', color='market', title='Mean target by day of week (0=Mon)')
fig.show()

In [ ]:
pivot_hour = train.pivot_table(index='market', columns='hour', values='target', aggfunc='mean')
fig = px.imshow(
    pivot_hour,
    aspect='auto',
    title='Market x Hour heatmap (mean target)',
    color_continuous_scale='RdBu_r'
)
fig.show()

In [ ]:
pivot_dow = train.pivot_table(index='market', columns='dow', values='target', aggfunc='mean')
fig = px.imshow(
    pivot_dow,
    aspect='auto',
    title='Market x Day-of-week heatmap (mean target)',
    color_continuous_scale='RdBu_r'
)
fig.show()

## 5) Numeric Feature Distribution Analysis

In [ ]:
exclude_cols = {'id', 'target', 'market', 'delivery_start', 'delivery_end'}
num_cols = [c for c in train.columns if c not in exclude_cols and pd.api.types.is_numeric_dtype(train[c])]
len(num_cols), num_cols[:10]

In [ ]:
# Summary stats table for numeric features
num_summary = train[num_cols].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T
num_summary['missing_ratio'] = train[num_cols].isna().mean()
num_summary = num_summary.sort_values('std', ascending=False)
display(num_summary)

In [ ]:
# Distribution plots in batches
batch_size = 6
for i in range(0, len(num_cols), batch_size):
    cols = num_cols[i:i+batch_size]
    n = len(cols)
    fig, axes = plt.subplots(nrows=n, ncols=2, figsize=(12, 3.4*n))
    if n == 1:
        axes = np.array([axes])

    for r, col in enumerate(cols):
        axes[r, 0].hist(train[col].dropna(), bins=60)
        axes[r, 0].set_title(f'{col} (hist)')

        x = train[col].dropna()
        if len(x) > 0:
            q1, q3 = np.percentile(x, [25, 75])
            iqr = q3 - q1
            low = q1 - 1.5*iqr
            high = q3 + 1.5*iqr
            clipped = x[(x >= low) & (x <= high)]
            axes[r, 1].hist(clipped, bins=60)
            axes[r, 1].set_title(f'{col} (IQR-clipped)')
        else:
            axes[r, 1].set_title(f'{col} (no data)')

    plt.tight_layout()
    plt.show()

In [ ]:
outlier_rows = []
for col in num_cols:
    x = train[col].dropna()
    if len(x) < 10:
        continue
    q1, q3 = np.percentile(x, [25, 75])
    iqr = q3 - q1
    low = q1 - 1.5 * iqr
    high = q3 + 1.5 * iqr
    out_ratio = ((train[col] < low) | (train[col] > high)).mean()
    outlier_rows.append((col, out_ratio, low, high))

outlier_df = pd.DataFrame(outlier_rows, columns=['feature', 'outlier_ratio', 'iqr_low', 'iqr_high'])
outlier_df = outlier_df.sort_values('outlier_ratio', ascending=False)
display(outlier_df.head(20))

## 6) Correlation and Interaction Analysis

In [ ]:
corr_cols = ['target'] + num_cols
corr = train[corr_cols].corr(numeric_only=True)

# Top absolute correlations to target
target_corr = corr['target'].drop('target').sort_values(key=lambda s: s.abs(), ascending=False)
display(target_corr.head(25).to_frame('corr_with_target'))

In [ ]:
# Heatmap of top features by absolute target correlation
top_feats = target_corr.head(20).index.tolist()
heat = train[['target'] + top_feats].corr(numeric_only=True)
fig = px.imshow(heat, aspect='auto', title='Correlation heatmap: target + top correlated features', color_continuous_scale='RdBu_r', zmin=-1, zmax=1)
fig.show()

In [ ]:
# Scatter views for top 6 correlated features
for col in target_corr.head(6).index:
    sample = train[[col, 'target', 'market']].dropna().sample(min(8000, train[[col, 'target']].dropna().shape[0]), random_state=42)
    fig = px.scatter(sample, x=col, y='target', color='market', opacity=0.45, title=f'{col} vs target (sampled)')
    fig.show()

## 7) Train vs Test Drift Checks

In [ ]:
def quantile_drift(train_s, test_s, q=10):
    # Absolute difference in quantile bins; simple and robust drift score.
    qs = np.linspace(0, 1, q + 1)
    tq = np.nanquantile(train_s, qs)
    eq = np.nanquantile(test_s, qs)
    return float(np.mean(np.abs(tq - eq)))

common_num = [c for c in num_cols if c in test.columns]
drift = []
for c in common_num:
    score = quantile_drift(train[c].to_numpy(), test[c].to_numpy(), q=20)
    drift.append((c, score, train[c].mean(), test[c].mean(), train[c].std(), test[c].std()))

drift_df = pd.DataFrame(drift, columns=['feature', 'quantile_drift', 'train_mean', 'test_mean', 'train_std', 'test_std'])
drift_df = drift_df.sort_values('quantile_drift', ascending=False)
display(drift_df.head(25))

In [ ]:
# Visualize top drifted features
for col in drift_df.head(8)['feature']:
    x1 = train[col].dropna()
    x2 = test[col].dropna()

    fig, ax = plt.subplots(figsize=(10, 3.5))
    ax.hist(x1, bins=70, alpha=0.55, density=True, label='train')
    ax.hist(x2, bins=70, alpha=0.55, density=True, label='test')
    ax.set_title(f'Train vs test distribution: {col}')
    ax.legend()
    plt.show()

## 8) Engineered Features (Aligned with `main.py`)

In [ ]:
def add_engineered_features(df):
    out = df.copy()

    out['delivery_duration_hours'] = (out['delivery_end'] - out['delivery_start']).dt.total_seconds() / 3600
    out['delivery_start_ts'] = out['delivery_start'].astype('int64') // 10**9

    if {'air_temperature_2m', 'dew_point_temperature_2m'}.issubset(out.columns):
        out['temp_dew_spread'] = out['air_temperature_2m'] - out['dew_point_temperature_2m']

    if {'air_temperature_2m', 'apparent_temperature_2m'}.issubset(out.columns):
        out['temp_apparent_spread'] = out['air_temperature_2m'] - out['apparent_temperature_2m']

    if {'wind_speed_80m', 'wind_speed_10m'}.issubset(out.columns):
        out['wind_speed_ratio_80m_10m'] = out['wind_speed_80m'] / (out['wind_speed_10m'].abs() + 1e-3)

    if {'global_horizontal_irradiance', 'diffuse_horizontal_irradiance', 'direct_normal_irradiance'}.issubset(out.columns):
        out['irradiance_total'] = (
            out['global_horizontal_irradiance']
            + out['diffuse_horizontal_irradiance']
            + out['direct_normal_irradiance']
        )

    if {'cloud_cover_total', 'cloud_cover_low', 'cloud_cover_mid', 'cloud_cover_high'}.issubset(out.columns):
        out['cloud_cover_layers_sum'] = out['cloud_cover_low'] + out['cloud_cover_mid'] + out['cloud_cover_high']
        out['cloud_cover_gap'] = out['cloud_cover_total'] - out['cloud_cover_layers_sum']

    return out

train_eng = add_engineered_features(train)

In [ ]:
eng_candidates = [
    'delivery_duration_hours',
    'temp_dew_spread',
    'temp_apparent_spread',
    'wind_speed_ratio_80m_10m',
    'irradiance_total',
    'cloud_cover_layers_sum',
    'cloud_cover_gap',
]
eng_cols = [c for c in eng_candidates if c in train_eng.columns]

eng_corr = train_eng[['target'] + eng_cols].corr(numeric_only=True)['target'].drop('target').sort_values(key=lambda s: s.abs(), ascending=False)
display(eng_corr.to_frame('corr_with_target'))

In [ ]:
for col in eng_cols:
    sample = train_eng[[col, 'target', 'market']].dropna().sample(min(8000, train_eng[[col, 'target']].dropna().shape[0]), random_state=42)
    fig = px.scatter(sample, x=col, y='target', color='market', opacity=0.45, title=f'Engineered feature: {col} vs target (sampled)')
    fig.show()

## 9) Market-Specific Deep Dives

Use this section to zoom into one market at a time when you want detailed behavior checks.

In [ ]:
market_pick = sorted(train['market'].unique())[0]
sub = train_eng[train_eng['market'] == market_pick].sort_values('delivery_start').copy()

print('Selected market:', market_pick)
print('Rows:', len(sub))

fig = px.line(sub, x='delivery_start', y='target', title=f'Target over time - {market_pick}')
fig.show()

fig = px.histogram(sub, x='target', nbins=80, title=f'Target distribution - {market_pick}', marginal='box')
fig.show()

In [ ]:
# Top feature correlations inside selected market
local_num = [c for c in sub.columns if pd.api.types.is_numeric_dtype(sub[c]) and c not in {'id'}]
local_corr = sub[local_num].corr(numeric_only=True)['target'].drop('target').sort_values(key=lambda s: s.abs(), ascending=False)
display(local_corr.head(20).to_frame('corr_with_target'))

## 10) Key Takeaways Template

After running all cells, summarize:
1. Which features are most predictive globally vs per market.
2. Whether target has heavy tails/spikes and where they occur.
3. Which markets have distinct behavior regimes.
4. Which columns show train-test drift requiring robust modeling.
5. Which engineered features add meaningful signal.